# 0.0. Setup

In [1]:
import os
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
DATA_PUBLIC_DIR = PROJECT_DIR / 'data/public'
DATA_PRIVATE_DIR = PROJECT_DIR / 'data/private'
DATA_PROCESSED_DIR = PROJECT_DIR / 'data/processed'
OUTPUT_DIR = PROJECT_DIR / 'output'

# 1.0. Datasets

## 1.1. Load

In [2]:
!ls {OUTPUT_DIR}

all_schools_coordinates_ncr_region4a.parquet
analysis_payload
archive
consolidated_geodata_matched.gpkg
grade_7_student_flow_table_sy2324.parquet
processed_esc_beneficiaries.parquet
processed_esc_slots.parquet
processed_gastpe_data.parquet
processed_grade_7_enrollment.parquet
processed_private_school_coordinates.parquet
processed_private_seats.parquet
processed_project_bukas_school_enrollment.parquet
processed_project_bukas_school_information.parquet
processed_project_bukas_school_information_psgc.parquet
processed_public_school_coordinates.parquet
processed_public_seats.parquet
processed_shs_offerings.parquet
province_road_networks
reports
road_network_ncr_region4a.geojson
school_distance_matrix_index.json
school_distance_matrix_index_8307.json
school_distance_matrix_osrm.npy
school_distance_matrix_osrm_8307.npy
scratch


In [3]:
fname = 'processed_grade_7_enrollment.parquet'
fpath = os.path.join(OUTPUT_DIR, fname)
df_g7 = pd.read_parquet(fpath)
print(df_g7.shape)

(6041288, 10)


In [4]:
fname = 'processed_esc_beneficiaries.parquet'
fpath = os.path.join(OUTPUT_DIR, fname)
df_benef = pd.read_parquet(fpath)
print(df_benef.shape)

(2681668, 14)


In [5]:
# School info
fname = 'processed_project_bukas_school_information.parquet'
fpath = os.path.join(OUTPUT_DIR, fname)
sch_info = pd.read_parquet(fpath)
print(sch_info.shape)

(61442, 20)


## 1.2. Inspect

In [6]:
display(df_g7.head(3))

,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,old_region,school_year
0,Region I,Ilocos Norte,100160150023,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023
1,Region I,Ilocos Norte,100160150024,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023
2,Region I,Ilocos Norte,100160150025,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023


In [7]:
df_g7['school_year'].unique()

['2022-2023', '2023-2024', '2024-2025']
Categories (3, object): ['2022-2023', '2023-2024', '2024-2025']

In [8]:
df_g7['old_region'].unique()

array(['Region I', 'Region VI', 'Region II', 'Region III', 'Region IV-A',
       'MIMAROPA', 'Region V', 'Region VII', 'Region VIII', 'Region IX',
       'Region X', 'Region XI', 'Region XII', 'CARAGA', 'BARMM', 'CAR',
       'NCR', 'PSO'], dtype=object)

In [9]:
display(df_benef.head(3))

,old_region,destination_school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,lrn_category,origin_school_id,is_valid_origin_school_id,school_year
0,BARMM,475511,133526130177,Mae Ann Canoy Abajo,1603520,"Adiong Memorial College Foundation, Inc.",grade_9,ESC-227153,9000,BARMM,Standard,133526,True,2022-2023
1,BARMM,475511,133536150012,MA. JOY PAGUNTALAN ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_7,ESC-227153,9000,BARMM,Standard,133536,True,2022-2023
2,BARMM,475511,133536150011,MA. GRACE PAGUNTALAN ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_7,ESC-227153,9000,BARMM,Standard,133536,True,2022-2023


In [10]:
df_benef['school_year'].unique()

['2022-2023', '2023-2024', '2024-2025']
Categories (3, object): ['2022-2023', '2023-2024', '2024-2025']

In [11]:
df_benef['old_region'].unique()

array(['BARMM', 'CAR', 'NCR', 'Region I', 'Region II', 'Region III',
       'Region IV-A', 'MIMAROPA', 'Region V', 'Region VI', 'Region VII',
       'Region VIII', 'Region IX', 'Region X', 'Region XI', 'Region XII',
       'CARAGA'], dtype=object)

In [12]:
df_benef['grade_level'].unique()

array(['grade_9', 'grade_7', 'grade_10', 'grade_8'], dtype=object)

## 1.3. Preprocess

### SY 23-24

In [13]:
g7 = df_g7.copy()
g7 = g7[g7['school_year'] == '2023-2024']
g7['is_esc_beneficiary'] = False
print(f"Shape of SY 23-24 Grade 7 enrollment flow: {g7.shape}\n")

bf = df_benef.copy()
bf['is_esc_beneficiary'] = True
bf = bf[
    (bf['school_year'] == '2023-2024')
    & (bf['grade_level'] == 'grade_7')
    & (bf['is_valid_origin_school_id'] == True)
]
bf.rename(
    columns={
        'destination_school_id':'school_id_destination',
        'origin_school_id':'school_id_origin',
    },
    inplace=True
)
print(f"Shape of SY 23-24 ESC Beneficiary flow: {bf.shape}")

# Info
bf_alt = df_benef[df_benef['school_year'] == '2023-2024']
len_invalid_og_ids = sum(bf_alt['is_valid_origin_school_id'] == False)
print(f"Count of invalid school IDs extracted from LRN is: {len_invalid_og_ids:,} out of {len(bf_alt):,}")

Shape of SY 23-24 Grade 7 enrollment flow: (1786442, 11)

Shape of SY 23-24 ESC Beneficiary flow: (209072, 15)
Count of invalid school IDs extracted from LRN is: 55,996 out of 897,016


### Non-beneficiaries

In [14]:
g7_lrns = g7['lrn'].unique().tolist()
benef_lrns = bf['lrn'].unique().tolist()

ix_g7_bf = set(g7_lrns).intersection(set(benef_lrns))
diff_g7_bf = set(g7_lrns).difference(set(benef_lrns))
len_ix = len(ix_g7_bf)
len_diff = len(diff_g7_bf)
print(f"{len_ix:,}; {len_diff:,}")

ix_bf_g7 = set(benef_lrns).intersection(set(g7_lrns))
len_ix = len(ix_bf_g7)
print(len_ix)

206,598; 1,579,844
206598


In [15]:
# We remove LRNs that appear in the ESC beneficiary dataset
non_bf = g7[~g7['lrn'].isin(benef_lrns)].copy()
print(non_bf.shape)

(1579844, 11)


In [16]:
display(non_bf.head(2))

,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,old_region,school_year,is_esc_beneficiary
2136862,Region I,Ilocos Norte,100160170039,300001,Adams National High School,2022,100160,Adams CES,Region I,2023-2024,False
2136863,Region I,Ilocos Norte,100160160033,300001,Adams National High School,2022,100160,Adams CES,Region I,2023-2024,False


In [17]:
# Review of benef dataframe
display(bf.head(2))

,old_region,school_id_destination,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,lrn_category,school_id_origin,is_valid_origin_school_id,school_year,is_esc_beneficiary
920901,BARMM,406057,475002160071,RANNIA JAKILAN ABANNI,1601792,Claret School of Lamitan,grade_7,ESC-243708,9000,BARMM,Standard,475002,True,2023-2024,True
920902,BARMM,406057,132985160071,HISHAM ABDULMAJID ABBAS,1601792,Claret School of Lamitan,grade_7,ESC-243708,9000,BARMM,Standard,132985,True,2023-2024,True


## 1.4. Aggregate

In [18]:
uniq_origin_schools = non_bf['school_id_origin'].unique()
len_uos = len(uniq_origin_schools)
print(f"Count of unique origin schools in non-beneficiary dataset: {len_uos:,}")

uniq_dest_schools = non_bf['school_id_destination'].unique()
len_uds = len(uniq_dest_schools)
print(f"Count of unique destination schools in non-beneficiary dataset: {len_uds:,}")

Count of unique origin schools in non-beneficiary dataset: 44,625
Count of unique destination schools in non-beneficiary dataset: 15,510


In [19]:
relevant_cols = ['school_id_origin','school_id_destination','lrn','is_esc_beneficiary']
df_agg = pd.concat(
    [
        non_bf[relevant_cols],
        bf[relevant_cols]
    ]
)
print(df_agg.shape)

(1788916, 4)


In [20]:
display(df_agg.head(2))
display(df_agg.tail(2))
display(df_agg['is_esc_beneficiary'].value_counts())

,school_id_origin,school_id_destination,lrn,is_esc_beneficiary
2136862,100160,300001,100160170039,False
2136863,100160,300001,100160160033,False


,school_id_origin,school_id_destination,lrn,is_esc_beneficiary
1817911,473512,406012,473512150021,True
1817912,109331,406012,109331160494,True


is_esc_beneficiary
False    1579844
True      209072
Name: count, dtype: int64

In [23]:
# School-level student flow
dfa = df_agg.groupby(['school_id_origin','school_id_destination','is_esc_beneficiary'])['lrn'].size().reset_index()
dfa.rename(
    columns={
        'lrn':'count_student'
    },
    inplace=True
)

# Sanity crosscheck with Bukas enrollment
pvt_dfa = dfa.pivot_table(
    index=['school_id_destination'],
    columns=['is_esc_beneficiary'],
    values='count_student',
    aggfunc='sum',
    observed=False
)

# Rename columns
pvt_dfa.rename(
    columns={
        True:'count_non_beneficiary',
        False:'count_esc_beneficiary',
    },
    inplace=True
)

print(pvt_dfa.shape)

(16035, 2)


In [24]:
display(dfa.head(3))

,school_id_origin,school_id_destination,is_esc_beneficiary,count_student
0,100000,300378,False,8
1,100000,300388,False,25
2,100000,305334,False,1


In [25]:
display(pvt_dfa.sample(5))

is_esc_beneficiary,count_esc_beneficiary,count_non_beneficiary
school_id_destination,,
303111,238.0,NaN
404683,NaN,53.0
406746,NaN,10.0
402883,2.0,11.0
409148,4.0,40.0


In [26]:
mask = pvt_dfa.apply(lambda row: all(True if el > 0 else False for el in row), axis=1)
pvt_dfa[mask].sample(5)

is_esc_beneficiary,count_esc_beneficiary,count_non_beneficiary
school_id_destination,,
400093,3.0,10.0
405044,7.0,119.0
404315,16.0,132.0
402529,25.0,63.0
402033,11.0,34.0


### Crosscheck

In [174]:
dirpath = 'output/scratch'
fname = 'processed_sy2324_enrollment.parquet'
outpath = os.path.join(dirpath, fname)

# ENROLLMENT
sub_enr = pd.read_parquet(outpath)
pvt_enr = sub_enr.pivot_table(
    index='school_id',
    columns='enrollment_label',
    values='enrollment_count',
    aggfunc='sum',
    observed=False,
)
print(pvt_enr.shape)
# display(pvt_enr.head(3))

(58810, 2)


In [158]:
# ESC BENEFICIARY
print(bf.shape)
display(bf.head(3))

(209072, 15)


,old_region,school_id_destination,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,lrn_category,school_id_origin,is_valid_origin_school_id,school_year,is_esc_beneficiary
920901,BARMM,406057,475002160071,RANNIA JAKILAN ABANNI,1601792,Claret School of Lamitan,grade_7,ESC-243708,9000,BARMM,Standard,475002,True,2023-2024,True
920902,BARMM,406057,132985160071,HISHAM ABDULMAJID ABBAS,1601792,Claret School of Lamitan,grade_7,ESC-243708,9000,BARMM,Standard,132985,True,2023-2024,True
920918,BARMM,406057,132988160048,NURLEEYANA MACROHON ABDUL,1601792,Claret School of Lamitan,grade_7,ESC-243708,9000,BARMM,Standard,132988,True,2023-2024,True


In [181]:
schids = ['501297','401574']

# Enrollment
display(pvt_enr[pvt_enr.index.isin(schids)])

# ESC Beneficiary
display(bf[bf['school_id_destination'].isin(schids)]['school_id_destination'].value_counts())

enrollment_label,enr_grade_6,enr_grade_7
school_id,,
401574,80.0,159.0
501297,31.0,20.0


school_id_destination
401574    151
Name: count, dtype: int64

# 2.0. Save

In [207]:
tmp_pvt = (
    dfa.pivot_table(
        index=['school_id_origin','school_id_destination'],
        columns='is_esc_beneficiary',
        values='count_student',
        aggfunc='sum',
        observed=False
    )
    .rename(
        columns={
            True:'count_esc_beneficiary',
            False:'count_non_beneficiary'
        }
    )
    .reset_index()
)
tmp_pvt.columns.name = ''

print(tmp_pvt.shape)

(288328, 4)


In [208]:
display(tmp_pvt.head())

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary
0,100000,300378,8.0,NaN
1,100000,300388,25.0,NaN
2,100000,305334,1.0,NaN
3,100000,500057,2.0,NaN
4,100001,300002,6.0,NaN


In [209]:
mask = tmp_pvt['school_id_destination'].isin(schids)
tmp_v = tmp_pvt[mask]
tmp_aggsum = tmp_v.groupby(['school_id_destination']).agg(
    non_benef=('count_non_beneficiary', 'sum'),
    esc_benef=('count_esc_beneficiary', 'sum')
)

# display(tmp_v)
display(tmp_aggsum)

,non_benef,esc_benef
school_id_destination,,
401574,9.0,151.0
501297,20.0,0.0


In [210]:
fpath = 'output/grade_7_student_flow_table_sy2324.parquet'
tmp_pvt.to_parquet(fpath)